# 03 — NLI Cross-Encoder (Zero-Shot)
**Roll No:** 23f3004491 | Model 3 of 5 | Milestone 2

A cross-encoder reads the prompt and an option together and scores how well the option follows
from the prompt (entailment). Because it attends over both texts jointly, it captures reasoning
a bi-encoder cannot.

In [ ]:
import warnings, re
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

BASE = "/kaggle/input/competitions/smart-mcq-solver-challenge"
train = pd.read_csv(f"{BASE}/train.csv")
test  = pd.read_csv(f"{BASE}/test.csv")
OPTIONS = ["A", "B", "C", "D", "E"]

def average_precision_at_3(true_label, predicted_labels):
    for rank, pred in enumerate(predicted_labels[:3]):
        if pred == true_label:
            return 1.0 / (rank + 1)
    return 0.0

def mean_average_precision_at_3(true_labels, predicted_lists):
    return float(np.mean([average_precision_at_3(t, p)
                          for t, p in zip(true_labels, predicted_lists)]))

START_WRAPPERS = ["Pick the best possible answer:", "Select the most accurate option:",
                  "Determine the correct option:", "Identify the correct statement:",
                  "Choose the correct answer:"]

def normalize_core(prompt):
    p = str(prompt).strip()
    for s in START_WRAPPERS:
        if p.startswith(s):
            p = p[len(s):].strip()
    if "?" in p:
        p = p[:p.rfind("?") + 1]
    return re.sub(r"\s+", " ", p).lower().strip()

train["core"] = train["prompt"].apply(normalize_core)
test["core"]  = test["prompt"].apply(normalize_core)

# leakage-free split by unique core question
np.random.seed(42)
cores = train["core"].unique().copy()
np.random.shuffle(cores)
val_cores = set(cores[:200])
valid_df = train[train["core"].isin(val_cores)].drop_duplicates("core").reset_index(drop=True)
print("train:", train.shape, "| validation questions:", len(valid_df))

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import CrossEncoder

def scores_to_preds(S):
    return [[OPTIONS[j] for j in np.argsort(row)[::-1]] for row in S]

ce_model = CrossEncoder("cross-encoder/nli-deberta-v3-small")

def cross_encoder_scores(df):
    pairs = [(row["prompt"], row[o]) for _, row in df.iterrows() for o in OPTIONS]
    logits = ce_model.predict(pairs, batch_size=32, show_progress_bar=False)
    entail = logits[:, 1] if logits.ndim == 2 else logits
    return entail.reshape(len(df), 5)

In [ ]:
preds = scores_to_preds(cross_encoder_scores(valid_df))
score = mean_average_precision_at_3(valid_df["answer"].tolist(), preds)
print(f"Cross-encoder validation MAP@3: {score:.4f}")

## Observation
The cross-encoder reaches about 0.56 zero-shot — the strongest neural model so far and the
best pure-reasoning signal available on CPU. In the final pipeline it serves as the rank-2
hedge behind the retrieval lookup.